# 🔧 Merge LoRA (rsLoRA) + Cuantización GGUF — Qwen3.5-0.8B

Este notebook toma un adaptador LoRA entrenado con **Unsloth** (rsLoRA) sobre
`unsloth/Qwen3.5-0.8B`, lo fusiona con el modelo base original y genera
archivos **GGUF** cuantizados, guardando todo en una carpeta de **Google Drive**.
Al final hay una celda opcional para subir los resultados a **Hugging Face Hub**.


## Paso 0 — Configuración

In [ ]:
# === CONFIGURACIÓN — EDITA ESTOS VALORES ===

# Ruta a la carpeta LoRA: puede ser la carpeta "LoRA only" o la carpeta raíz
# de checkpoints (en cuyo caso se detecta automáticamente el más reciente).
LORA_ADAPTER_PATH = "/content/drive/MyDrive/ruta/a/tu/carpeta_lora"

# Modelo base original (se descargará automáticamente desde Hugging Face)
BASE_MODEL_NAME = "unsloth/Qwen3.5-0.8B"

# Carpeta de Google Drive donde se guardarán los resultados (se crea si no existe)
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/qwen3.5-0.8b"

# Métodos de cuantización a generar. Algunos valores válidos:
# "q2_k", "q3_k_m", "q4_0", "q4_k_s", "q4_k_m", "q5_k_m", "q6_k", "q8_0", "f16", "bf16"
# q4_k_m = buen balance tamaño/calidad. q8_0 = casi sin pérdida, más pesado.
QUANT_METHODS = ["q4_k_m", "q8_0"]

# El cabezal MTP (Multi-Token Prediction) de Qwen3.5 sirve para acelerar la
# inferencia vía "speculative decoding", pero requiere banderas especiales en
# llama.cpp/llama-server y un modelo "draft" aparte. Para un despliegue simple
# (un único .gguf que funcione en cualquier lado: Ollama, LM Studio, llama.cpp
# normal) lo recomendado es dejarlo en False.
INCLUDE_MTP_HEAD = False

# Si True, además de los .gguf, guarda en Drive el modelo fusionado en formato
# HF/safetensors de 16-bit (útil para subirlo a HF Hub o re-cuantizar después
# sin tener que rehacer el merge).
KEEP_MERGED_SAFETENSORS_IN_DRIVE = True

# Si True, conserva en Drive el .gguf intermedio en f16 (sin cuantizar) además
# de los cuantizados. Para un modelo de 0.8B el costo en espacio es bajo (~1.6GB).
KEEP_F16_GGUF = True

# Longitud máxima de secuencia (solo se usa para la prueba rápida de inferencia,
# no afecta el merge en sí).
MAX_SEQ_LENGTH = 2048

## Paso 1 — Montar Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print("Carpeta de salida en Drive:", DRIVE_OUTPUT_DIR)

## Paso 2 — Instalar dependencias

Instalamos Unsloth (para cargar el modelo base correctamente) y forzamos una
versión reciente de `transformers`, ya que Qwen3.5 es un lanzamiento reciente
y necesita soporte actualizado en la librería.

In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

In [ ]:
%%capture
!pip install -q --upgrade unsloth unsloth_zoo
!pip install -q --upgrade "transformers>=5.3.0"
!pip install -q --upgrade peft accelerate huggingface_hub hf_transfer sentencepiece protobuf gguf

!pip uninstall -y torchaudio -q

In [ ]:
# Verificación rápida de versiones instaladas
import transformers
import peft
import torch
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("torch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

## Paso 3 — Resolver la carpeta del adaptador LoRA

Si `LORA_ADAPTER_PATH` apunta directamente a una carpeta con `adapter_config.json`
(p. ej. tu export "LoRA only"), se usa tal cual. Si apunta a una carpeta con
varios `checkpoint-XXXX/`, se detecta automáticamente el checkpoint más reciente
que tenga un adaptador válido.

In [ ]:
import os
import glob


def resolve_adapter_dir(path):
    if os.path.isfile(os.path.join(path, "adapter_config.json")):
        return path

    checkpoints = glob.glob(os.path.join(path, "checkpoint-*"))

    def ckpt_num(p):
        try:
            return int(p.rstrip("/").split("-")[-1])
        except ValueError:
            return -1
    checkpoints = sorted(checkpoints, key=ckpt_num)
    valid = [c for c in checkpoints if os.path.isfile(
        os.path.join(c, "adapter_config.json"))]

    if not valid:
        raise FileNotFoundError(
            f"No se encontró adapter_config.json en '{path}' ni en sus subcarpetas checkpoint-*. "
            "Verifica que LORA_ADAPTER_PATH sea correcto."
        )
    latest = valid[-1]
    print(
        f"Se encontraron {len(valid)} checkpoint(s) válidos. Usando el más reciente: {latest}")
    return latest


ADAPTER_DIR = resolve_adapter_dir(LORA_ADAPTER_PATH)
print("Adaptador LoRA resuelto en:", ADAPTER_DIR)
print("\nArchivos en el adaptador:")
for f in sorted(os.listdir(ADAPTER_DIR)):
    print(" -", f)

## Paso 4 — Descargar y cargar el modelo base (16-bit, sin cuantizar)

Usamos `FastModel` de Unsloth (no `FastLanguageModel`) porque Qwen3.5 está
registrado como una arquitectura de visión-lenguaje unificada, y `FastModel` es
la clase que Unsloth recomienda para este tipo de modelos incluso en uso solo-texto.

Cargamos en **bf16 sin cuantizar** (`load_in_4bit=False`) para que el merge sea
limpio — Unsloth desaconseja entrenar/fusionar Qwen3.5 en 4-bit por diferencias
de precisión más altas de lo normal en este modelo.

In [ ]:
import torch
from unsloth import FastModel

print(f"Descargando / cargando el modelo base: {BASE_MODEL_NAME}")
print("(esto puede tardar unos minutos la primera vez)")

base_model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_8bit=False,
    dtype=torch.bfloat16,
)
print("\n✅ Modelo base cargado.")

## Paso 5 — Cargar el adaptador y fusionar (merge)

Adjuntamos el adaptador con `peft.PeftModel` (no con las funciones de alto nivel
de Unsloth) y fusionamos con `merge_and_unload()`. Esto respeta automáticamente
la configuración de **rsLoRA** guardada en `adapter_config.json` (rank, alpha,
`use_rslora`), no necesitas indicar nada manualmente.

In [ ]:
import json
from peft import PeftModel

with open(os.path.join(ADAPTER_DIR, "adapter_config.json")) as f:
    adapter_cfg = json.load(f)

print("Configuración del adaptador detectada:")
print("  r:            ", adapter_cfg.get("r"))
print("  lora_alpha:   ", adapter_cfg.get("lora_alpha"))
print("  use_rslora:   ", adapter_cfg.get("use_rslora"))
print("  target_modules:", adapter_cfg.get("target_modules"))

print("\nCargando adaptador LoRA sobre el modelo base...")
peft_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

print("Fusionando adaptador con el modelo base (merge_and_unload)...")
merged_model = peft_model.merge_and_unload()

# Aseguramos que todo quede en bf16 (el merge a veces deja algún tensor en fp32)
merged_model = merged_model.to(torch.bfloat16)

print("✅ Merge completado.")

## Paso 6 — Guardar el modelo fusionado (formato HF / safetensors)

Guardamos primero en almacenamiento local de Colab (más rápido) y luego copiamos
a Drive. También copiamos cualquier archivo de tokenizer/chat-template que venga
en la carpeta del adaptador, por si tu fine-tuning agregó tokens especiales o
modificó el chat template — así evitamos el problema típico de "el modelo
funciona bien en Unsloth pero da resultados raros en otra plataforma" causado
por usar un chat template distinto al de entrenamiento.

In [ ]:
import shutil

LOCAL_MERGED_DIR = "/content/merged_model_16bit"
os.makedirs(LOCAL_MERGED_DIR, exist_ok=True)

print("Guardando modelo fusionado en formato HF (16-bit)...")
merged_model.save_pretrained(LOCAL_MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(LOCAL_MERGED_DIR)

# Preservar posibles cambios de tokenizer/chat-template hechos durante el fine-tuning
preserve_files = [
    "tokenizer_config.json", "chat_template.jinja", "chat_template.json",
    "special_tokens_map.json", "added_tokens.json",
]
for fname in preserve_files:
    src = os.path.join(ADAPTER_DIR, fname)
    if os.path.isfile(src):
        shutil.copy(src, LOCAL_MERGED_DIR)
        print(
            f"  Copiado {fname} desde el adaptador (preserva el chat template de entrenamiento).")

print("\n✅ Modelo fusionado guardado localmente en:", LOCAL_MERGED_DIR)
for f in sorted(os.listdir(LOCAL_MERGED_DIR)):
    size_mb = os.path.getsize(os.path.join(LOCAL_MERGED_DIR, f)) / (1024**2)
    print(f"   - {f} ({size_mb:.1f} MB)")

In [ ]:
if KEEP_MERGED_SAFETENSORS_IN_DRIVE:
    drive_merged_dir = os.path.join(DRIVE_OUTPUT_DIR, "merged_16bit")
    os.makedirs(drive_merged_dir, exist_ok=True)
    print("Copiando modelo fusionado a Google Drive...")
    for fname in os.listdir(LOCAL_MERGED_DIR):
        shutil.copy(os.path.join(LOCAL_MERGED_DIR, fname),
                    os.path.join(drive_merged_dir, fname))
    print("✅ Copiado a:", drive_merged_dir)
else:
    print("KEEP_MERGED_SAFETENSORS_IN_DRIVE = False — se omite la copia a Drive del modelo 16-bit.")

## Paso 7 (opcional) — Prueba rápida del modelo fusionado

Una generación corta para confirmar que el merge produce texto coherente antes
de la conversión a GGUF.

In [ ]:
try:
    FastModel.for_inference(merged_model)

    mensajes = [{"role": "user", "content": "Hola, preséntate brevemente."}]
    input_ids = tokenizer.apply_chat_template(
        mensajes,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(merged_model.device)

    output_ids = merged_model.generate(
        input_ids=input_ids,
        max_new_tokens=64,
        use_cache=True,
        do_sample=False,
    )
    respuesta = tokenizer.decode(
        output_ids[0][input_ids.shape[-1]:], skip_special_tokens=True)
    print("Respuesta del modelo fusionado:\n")
    print(respuesta)
except Exception as e:
    print("⚠️ No se pudo correr la prueba de inferencia (no es crítico, puedes continuar):")
    print(e)

## Paso 8 — Liberar memoria antes de la conversión

No necesitamos GPU para convertir/cuantizar a GGUF (son procesos basados en CPU),
así que liberamos la memoria del modelo cargado.

In [ ]:
import gc

del peft_model
del base_model
del merged_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✅ Memoria liberada.")

## Paso 9 — Clonar y preparar `llama.cpp` (versión actualizada)

Clonamos `llama.cpp` directamente desde GitHub en lugar de usar la copia
empaquetada por Unsloth, para asegurarnos de tener el soporte más reciente de
la arquitectura Qwen3.5 y las banderas `--mtp` / `--no-mtp`.

Solo compilamos el binario `llama-quantize` (no hace falta CUDA ni compilar todo
el proyecto) — la conversión a GGUF en sí (`convert_hf_to_gguf.py`) es un script
de Python puro que no requiere compilación.

In [ ]:
%%capture
%cd /content
!rm -rf llama.cpp
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
%cd /content/llama.cpp
!pip install -q -r requirements.txt
!apt-get -qq update
!apt-get -qq install -y cmake build-essential


In [ ]:
%cd /content/llama.cpp
!cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release
!cmake --build build --config Release -j$(nproc) --target llama-quantize


In [ ]:
import os
quantize_bin = "/content/llama.cpp/build/bin/llama-quantize"
assert os.path.isfile(
    quantize_bin), "No se encontró el binario llama-quantize. Revisa la celda de compilación anterior."
print("✅ llama-quantize compilado correctamente en:", quantize_bin)

## Paso 10 — Convertir el modelo fusionado a GGUF (f16)

Por defecto excluimos el cabezal MTP (`--no-mtp`) para obtener un único `.gguf`
simple y compatible con cualquier runtime (Ollama, LM Studio, llama.cpp estándar).
Si pusiste `INCLUDE_MTP_HEAD = True` en el Paso 0, también se exporta un segundo
archivo `mtp-*.gguf` con el cabezal de predicción multi-token para *speculative
decoding* (requiere `--spec-type draft-mtp` al servir el modelo con `llama-server`).

In [ ]:
# Fix: transformers>=5.3.0 escribió "tokenizer_class": "TokenizersBackend" en
# tokenizer_config.json, una clase interna que AutoTokenizer no puede resolver.
# La reemplazamos por la clase genérica PreTrainedTokenizerFast, que carga
# perfectamente el tokenizer a partir de tokenizer.json (ya presente en la carpeta).
import json

tok_cfg_path = os.path.join(LOCAL_MERGED_DIR, "tokenizer_config.json")
with open(tok_cfg_path) as f:
    tok_cfg = json.load(f)

if tok_cfg.get("tokenizer_class") == "TokenizersBackend":
    tok_cfg["tokenizer_class"] = "PreTrainedTokenizerFast"
    with open(tok_cfg_path, "w") as f:
        json.dump(tok_cfg, f, indent=2, ensure_ascii=False)
    print("✅ tokenizer_class corregido a PreTrainedTokenizerFast")
else:
    print("tokenizer_class ya es:", tok_cfg.get(
        "tokenizer_class"), "- no se modificó nada.")

In [ ]:
import subprocess
import sys

GGUF_LOCAL_DIR = "/content/gguf_output"
os.makedirs(GGUF_LOCAL_DIR, exist_ok=True)

f16_path = os.path.join(GGUF_LOCAL_DIR, "model-F16.gguf")

convert_cmd = [
    sys.executable, "/content/llama.cpp/convert_hf_to_gguf.py",
    LOCAL_MERGED_DIR,
    "--outfile", f16_path,
    "--outtype", "f16",
]
if not INCLUDE_MTP_HEAD:
    convert_cmd.append("--no-mtp")

print("Ejecutando conversión a GGUF (f16)...")
print(" ".join(convert_cmd))
subprocess.run(convert_cmd, check=True)
print("\n✅ Conversión f16 completada:", f16_path)

mtp_path = None
if INCLUDE_MTP_HEAD:
    mtp_path = os.path.join(GGUF_LOCAL_DIR, "mtp-model-F16.gguf")
    mtp_cmd = [
        sys.executable, "/content/llama.cpp/convert_hf_to_gguf.py",
        LOCAL_MERGED_DIR,
        "--outfile", mtp_path,
        "--outtype", "f16",
        "--mtp",
    ]
    print("\nExportando cabezal MTP por separado...")
    subprocess.run(mtp_cmd, check=True)
    print("✅ GGUF del cabezal MTP guardado en:", mtp_path)

## Paso 11 — Cuantizar

Genera un `.gguf` por cada método indicado en `QUANT_METHODS` (Paso 0).

In [ ]:
quantized_paths = {}

for method in QUANT_METHODS:
    method_upper = method.upper()
    out_path = os.path.join(GGUF_LOCAL_DIR, f"model-{method_upper}.gguf")
    print(f"Cuantizando a {method_upper}...")
    subprocess.run([quantize_bin, f16_path, out_path,
                   method_upper], check=True)
    quantized_paths[method] = out_path
    size_gb = os.path.getsize(out_path) / (1024**3)
    print(f"  ✅ Listo: {out_path} ({size_gb:.2f} GB)\n")

if not KEEP_F16_GGUF:
    os.remove(f16_path)
    print("Archivo F16 intermedio eliminado (KEEP_F16_GGUF = False).")
else:
    print("Se conserva el archivo F16 intermedio (KEEP_F16_GGUF = True).")

## Paso 12 — Copiar resultados GGUF a Google Drive

In [ ]:
drive_gguf_dir = os.path.join(DRIVE_OUTPUT_DIR, "gguf")
os.makedirs(drive_gguf_dir, exist_ok=True)

for fname in os.listdir(GGUF_LOCAL_DIR):
    src = os.path.join(GGUF_LOCAL_DIR, fname)
    dst = os.path.join(drive_gguf_dir, fname)
    print(f"Copiando {fname} a Google Drive...")
    shutil.copy(src, dst)

print("\n" + "="*60)
print("✅ RESUMEN")
print("="*60)
if KEEP_MERGED_SAFETENSORS_IN_DRIVE:
    print("Modelo fusionado (16-bit, HF):",
          os.path.join(DRIVE_OUTPUT_DIR, "merged_16bit"))
print("Archivos GGUF en:", drive_gguf_dir)
for f in sorted(os.listdir(drive_gguf_dir)):
    full = os.path.join(drive_gguf_dir, f)
    size_gb = os.path.getsize(full) / (1024**3)
    print(f"   - {f} ({size_gb:.2f} GB)")

---
## Paso 13 (opcional) — Subir a Hugging Face Hub

In [ ]:
# === CONFIGURACIÓN DE LA SUBIDA — EDITA ESTOS VALORES ===

# Sube el modelo fusionado en 16-bit (formato HF / safetensors)
UPLOAD_MERGED_MODEL = False
UPLOAD_GGUF = True            # Sube los archivos .gguf cuantizados

# Cambia esto por tu_usuario/nombre-del-repo en HF
HF_REPO_ID = "tu_usuario/tu_repo"
HF_PRIVATE = True                    # True = repo privado, False = público

In [ ]:
from huggingface_hub import login, HfApi, create_repo
from getpass import getpass

hf_token = getpass(
    "Pega tu token de Hugging Face (con permiso de escritura — https://huggingface.co/settings/tokens): ")
login(token=hf_token)

api = HfApi()
create_repo(HF_REPO_ID, token=hf_token, private=HF_PRIVATE,
            exist_ok=True, repo_type="model")
print(f"Repositorio listo: https://huggingface.co/{HF_REPO_ID}")

if UPLOAD_MERGED_MODEL:
    print("\nSubiendo modelo fusionado (16-bit)...")
    api.upload_folder(
        folder_path=LOCAL_MERGED_DIR,
        repo_id=HF_REPO_ID,
        path_in_repo=".",
        token=hf_token,
        commit_message="Upload merged 16-bit model (LoRA fusionado con Qwen3.5-0.8B)",
    )
    print("✅ Modelo fusionado subido.")

if UPLOAD_GGUF:
    print("\nSubiendo archivos GGUF...")
    for fname in sorted(os.listdir(GGUF_LOCAL_DIR)):
        local_path = os.path.join(GGUF_LOCAL_DIR, fname)
        print(f"  Subiendo {fname}...")
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=fname,
            repo_id=HF_REPO_ID,
            token=hf_token,
            commit_message=f"Add {fname}",
        )
    print("✅ Archivos GGUF subidos.")

print(f"\n🎉 Listo: https://huggingface.co/{HF_REPO_ID}")